In [14]:
#Bibliotecas Fase 1
import os
import cv2
import pandas as pd
from hand_landmark_extractor import HandLandmarkExtractor

#Bibliotecas Fase 2
import numpy as np
import time
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, log_loss
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb

---

## Fase 1 - Preprocessamento e extração de Landmarks

In [15]:
dataset_path = "SignAlphaSet"
output_csv = "asl_landmarks_dataset.csv"

extractor = HandLandmarkExtractor(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.5,
    suppress_warnings=True
)

all_data_frames = []

# Verificar que dataset ainda não existe
if os.path.exists(output_csv):
    print("► Dataset de landmarks já existe!")
else:
    # 1. Iterar sobre todas as pastas do dataset SignAlphaSet (A-Z)
    if not os.path.exists(dataset_path):
        print(f"Erro: A pasta {dataset_path} não foi encontrada.")
        exit()
    
    folders = sorted([f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))])
    
    for label in folders:
        folder_path = os.path.join(dataset_path, label)
        print(f"A processar letra: {label}...")
    
        # Iterar sobre todas as imagens na pasta da letra
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)
            
            # Carrega a imagem
            image = cv2.imread(img_path)
            if image is None:
                continue
    
            # 2. Extrair os landmarks com a biblioteca HandLandmarkExtractor, utilizando lógica do test_camera.py
            hands_data = extractor.process_image_landmarks(image)
    
            if hands_data:
                # 3. Cada landmark contém informação da mão e coordenadas (x, y, z)
                # O método hands_data_to_dataframe já aplana os 21 pontos em 63 colunas + identificação da mão
                df_temp = extractor.hands_data_to_dataframe(hands_data)
                
                # 4. A letra correspondente (label) obtém-se através da pasta
                df_temp['label'] = label
                
                all_data_frames.append(df_temp)
    
    # Resultado Esperado: Um novo dataset estruturado
    if all_data_frames:
        final_df = pd.concat(all_data_frames, ignore_index=True)
        final_df.to_csv(output_csv, index=False)
        print(f"Dataset criado com sucesso: {output_csv}")
        print(f"Total de registos: {len(final_df)}")
    else:
        print("Nenhum landmark foi detetado nas imagens.")
    
    extractor.close()

A processar letra: A...
A processar letra: B...
A processar letra: C...
A processar letra: D...
A processar letra: E...
A processar letra: F...
A processar letra: G...
A processar letra: H...
A processar letra: I...
A processar letra: J...
A processar letra: K...
A processar letra: L...
A processar letra: M...
A processar letra: N...
A processar letra: O...
A processar letra: P...
A processar letra: Q...
A processar letra: R...
A processar letra: S...
A processar letra: T...
A processar letra: U...
A processar letra: V...
A processar letra: W...
A processar letra: X...
A processar letra: Y...
A processar letra: Z...
Dataset criado com sucesso: asl_landmarks_dataset.csv
Total de registos: 25566


---

## Fase 2 - Desenvolvimento e Treino do Modelo de Classificação

In [ ]:
def preprocess_landmarks(df, drop_motion_letters=True):
    df = df.copy()

    # 1. Remover letras com movimento (J e Z) conforme o planeado
    if drop_motion_letters and "label" in df.columns:
        df = df[~df["label"].isin(["J", "Z"])].reset_index(drop=True)

    # Identificar colunas de coordenadas
    x_cols = [c for c in df.columns if c.endswith("_x")]
    y_cols = [c for c in df.columns if c.endswith("_y")]
    z_cols = [c for c in df.columns if c.endswith("_z")]
    lm_cols = x_cols + y_cols + z_cols

    # 2. Espelhar mão esquerda (x = 1 - x) para unificar o referencial
    if "hand" in df.columns:
        is_left = df["hand"].astype(str).str.lower() == "left"
        df.loc[is_left, x_cols] = 1.0 - df.loc[is_left, x_cols]
        # Converter hand para feature binária
        df["is_right"] = (~is_left).astype(int)
        df = df.drop(columns=["hand"])

    # 3. Normalização: Centrar no WRIST e escalar pela distância WRIST -> MIDDLE_FINGER_MCP
    # Isto torna o modelo invariante à posição e ao tamanho da mão na câmara
    df[x_cols] = df[x_cols].sub(df["WRIST_x"], axis=0)
    df[y_cols] = df[y_cols].sub(df["WRIST_y"], axis=0)
    df[z_cols] = df[z_cols].sub(df["WRIST_z"], axis=0)

    scale = np.sqrt(df["MIDDLE_FINGER_MCP_x"]**2 + df["MIDDLE_FINGER_MCP_y"]**2 + df["MIDDLE_FINGER_MCP_z"]**2)
    df[lm_cols] = df[lm_cols].div(scale.replace(0, 1), axis=0)

    return df

def train_and_optimize(csv_path="asl_landmarks_dataset.csv"):
    df = pd.read_csv(csv_path)
    df = preprocess_landmarks(df)

    X = df.drop(columns=["label"])
    y = df["label"]
    
    #Enconding das Labels
    classes = sorted(df['label'].unique())
    
    # Criar o dicionário de mapeamento usando enumerate (Letra -> Número)
    label_map = {label: i for i, label in enumerate(classes)}
    # Criar o mapeamento inverso para usar na API (Número -> Letra)
    inv_label_map = {i: label for label, i in label_map.items()}
    
    print(f"Mapeamento gerado: {label_map}")
    
    # Aplicar o mapeamento à coluna label
    y_encoded = df['label'].map(label_map).values

    # Split 70/15/15
    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y_encoded, test_size=0.15, random_state=42, stratify=y_encoded)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.176, random_state=42, stratify=y_train_val)

    models_config = {
        "KNN": {
            "model": KNeighborsClassifier(),
            "params": {
                "n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21],
                "weights": ["uniform", "distance"],
                "metric": ["euclidean", "manhattan", "minkowski"],
                "p": [1],
                "algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
                "leaf_size": [10, 20]
            }
        }
    }

    best_model = None
    best_f1 = -1

    for name, cfg in models_config.items():
        print(f"Otimizando {name}...")
        grid = GridSearchCV(cfg["model"], cfg["params"], cv=3, n_jobs=-1, scoring='f1_weighted', verbose=3)
        grid.fit(X_train, y_train)
        
        val_preds = grid.predict(X_val)
        val_f1 = f1_score(y_val, val_preds, average='weighted')
        print(f"{name} - Val F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model = grid.best_estimator_
            best_name = name

    # Avaliação Final e Exportação
    print(f"\n--- MELHOR MODELO: {best_name} ---")
    test_preds = best_model.predict(X_test)
    
    print(classification_report(y_test, test_preds, target_names=classes))

    with open('melhor_modelo.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    with open('label_map.pkl', 'wb') as f:
        pickle.dump(inv_label_map, f)
    
    return best_model

train_and_optimize()

Mapeamento gerado: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10, 'M': 11, 'N': 12, 'O': 13, 'P': 14, 'Q': 15, 'R': 16, 'S': 17, 'T': 18, 'U': 19, 'V': 20, 'W': 21, 'X': 22, 'Y': 23}
Otimizando KNN...
Fitting 3 folds for each of 384 candidates, totalling 1152 fits


## Accuracy (Acurácia)

<center>
<span style="font-size: 24px;">
$\text{Accuracy} = \frac{\text{nº de previsões corretas}}{\text{nº total de previsões}}$
</span>
</center>


A métrica principal utilizada foi a acurácia, uma vez que o problema consiste numa classificação multiclasse com classes equilibradas, onde todas as letras têm igual importância.

    models_config = {
        'SVM': {
            'model': SVC(probability=True, random_state=42),
            'params': {
                'C': [1, 10],
                'kernel': ['rbf', 'linear']
            }
        },
        'XGBoost': {
            'model': xgb.XGBClassifier(random_state=42, eval_metric='mlogloss'),
            'params': {
                'n_estimators': [100, 200],
                'max_depth': [3, 6],
                'learning_rate': [0.1, 0.2]
            }
        },
        'NeuralNetwork': {
            'model': MLPClassifier(random_state=42, max_iter=500),
            'params': {
                'hidden_layer_sizes': [(128, 64), (64, 64)],
                'alpha': [0.0001, 0.001]
            }
        },
        'RandomForest': {
            'model': RandomForestClassifier(random_state=42),
            'params': {
                'n_estimators': [100, 200],
                'max_depth': [None, 20]
            }
        }
    }



          "KNN": {
            "model": KNeighborsClassifier(),
            "params": {
                "n_neighbors": [3, 5, 7, 9, 11],
                "weights": ["uniform", "distance"],
                "metric": ["euclidean", "manhattan", "minkowski"],
                "leaf_size": [20, 30, 40]
            }
        },
        "XGBoost": {
            "model": xgb.XGBClassifier(
                random_state=42, 
                eval_metric="mlogloss",
                tree_method="hist",
                n_jobs=-1 # Mais rápido para datasets grandes
            ),
            "params": {
                "n_estimators": [100, 300],
                "max_depth": [3, 6],
                "learning_rate": [0.1, 0.2],
                "subsample": [0.8, 1.0],
                "colsample_bytree": [0.8, 1.0],
                #"gamma": [0, 0.1, 0.2],  # Regularização para evitar overfitting
                #"min_child_weight": [1, 3, 5]
            }
        },
        "SVM": {
            "model": SVC(probability=True, random_state=42),
            "params": {
                "C": [0.1, 1, 10, 100],
                "kernel": ["rbf", "poly", "linear"],
                "gamma": ["scale", "auto", 0.01, 0.001],
                "degree": [3, 5]  # Apenas relevante se o kernel for 'poly'
            }
        }
    }